In [ ]:
pip install lightgbm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 11.4 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.metrics import (accuracy_score,
                             log_loss)
from sklearn.ensemble import \
     (RandomForestRegressor as RF,
      GradientBoostingRegressor as GBR)
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
import lightgbm as lgb
import timeit

In [6]:
df = pd.read_csv('Chen_Final_Result.csv', index_col=0, parse_dates=True)
df.head()
# Replace 'malignant' with 1 and 'benign' with 0
df = df.query("condition != 'black'")
# Multiple categories become 1
df['condition'] = df['condition'].replace({
    'neutral': 0,
    'positive': 1,
    'negative': 2
})
print(df['condition'].value_counts(dropna=False))

condition
0    138
1    138
2    138
Name: count, dtype: int64


/tmp/ipykernel_3798/212869613.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df = pd.read_csv('Chen_Final_Result.csv', index_col=0, parse_dates=True)
/tmp/ipykernel_3798/212869613.py:6: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['condition'] = df['condition'].replace({


In [ ]:
df.head()

,X,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,skewness_raw,mean_raw,Mean_Tonic,Max_Tonic,Min_Tonic,SD_Tonic,Mean_Phasic,Max_Phasic,Min_Phasic,SD_Phasic
Unnamed: 0,,,,,,,,,,,,,,,,,,,,,
1,SUD_003_01_EDA_10s,0.356146,0.36127,0.354865,0.354865,0.353584,0.353584,0.352302,0.352302,0.351021,...,-3.042697,0.318193,0.322706,0.345543,0.312532,0.010824,-0.004513,0.028255,-0.300436,0.079186
1,SUD_003_01_EDA_10s,0.356146,0.36127,0.354865,0.354865,0.353584,0.353584,0.352302,0.352302,0.351021,...,-3.042697,0.318193,0.322706,0.345543,0.312532,0.010824,0.001377,0.003736,-0.001767,0.001332
1,SUD_003_01_EDA_10s,0.356146,0.36127,0.354865,0.354865,0.353584,0.353584,0.352302,0.352302,0.351021,...,-3.042697,0.318193,0.322706,0.345543,0.312532,0.010824,0.000180,0.003020,-0.002561,0.001433
1,SUD_003_01_EDA_10s,0.356146,0.36127,0.354865,0.354865,0.353584,0.353584,0.352302,0.352302,0.351021,...,-3.042697,0.318193,0.322706,0.345543,0.312532,0.010824,0.000463,0.005081,-0.005588,0.002750
1,SUD_003_01_EDA_10s,0.356146,0.36127,0.354865,0.354865,0.353584,0.353584,0.352302,0.352302,0.351021,...,-3.042697,0.318193,0.322706,0.345543,0.312532,0.010824,-0.007194,0.002772,-0.021215,0.007016


In [7]:
cols_to_scale = df.columns.difference(['condition', 'source_rows', 'X'])
X = df[cols_to_scale]

# 2. Define your target (y)
# Fixed the typo from 'condtion' to 'condition'
Y = df['condition']
#Split test and train prior to standardization to prevent data leakage
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42, stratify=Y)

#Do not do log, do z score for all variables
from sklearn.preprocessing import StandardScaler

# 1. Get a list of all columns EXCEPT 'condition'
cols_to_scale = df.columns.difference(['condition', 'source_rows', 'X'])

# 2. Initialize and apply the scaler only to those columns
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)

# 4. Transform TEST
# Use the mean and std from X_train to scale X_test
X_test = scaler.transform(X_test)

# Now df has standardized features, but 'condition' remains 0/1 (or its original state)

In [10]:
start_time = timeit.default_timer()

condition_col = 'condition'  # rename the variable to avoid collision
colnames = df.drop(columns=[condition_col, 'source_rows']).columns.tolist()
#Logistic Regression for identifying significant vectors, currently broken
formula = f'{condition_col} ~ {" + ".join(colnames)}'
model = smf.glm(formula=formula, data=df, family=sm.families.Binomial())

result = model.fit()
print(result.summary())

# Print computational time and accuracy
elapsed = timeit.default_timer() - start_time
print("--- Run time is %s seconds ---" % elapsed)

/usr/local/lib/python3.12/dist-packages/statsmodels/genmod/families/links.py:198: RuntimeWarning: overflow encountered in exp
  t = np.exp(-z)


                 Generalized Linear Model Regression Results                  
Dep. Variable:              condition   No. Observations:                  368
Model:                            GLM   Df Residuals:                        0
Model Family:                Binomial   Df Model:                          367
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                    nan
Date:                Tue, 21 Apr 2026   Deviance:                       8608.9
Time:                        04:16:59   Pearson chi2:                 5.18e+17
No. Iterations:                     6   Pseudo R-squ. (CS):                nan
Covariance Type:            nonrobust                                         
                               coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------
Intercept               

/usr/local/lib/python3.12/dist-packages/statsmodels/genmod/families/family.py:1056: RuntimeWarning: divide by zero encountered in log
  special.gammaln(n - y + 1) + y * np.log(mu / (1 - mu + 1e-20)) +
/usr/local/lib/python3.12/dist-packages/statsmodels/genmod/families/family.py:1056: RuntimeWarning: invalid value encountered in multiply
  special.gammaln(n - y + 1) + y * np.log(mu / (1 - mu + 1e-20)) +


In [8]:
# Define start time to calculate computation time
start_time = timeit.default_timer()


# Initialize the LightGBM model
model = lgb.LGBMClassifier(learning_rate=0.09, max_depth=-5, random_state=42)

# Train the model
model.fit(X_train, y_train, eval_set=[(X_test, y_test)], eval_metric='logloss')

# Predict on the test set
y_pred = model.predict(X_test)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
#Cross validation
scores = cross_val_score(model, X_train, y_train, cv=5)

print(f"Scores per fold: {scores}")
print(f"Mean Accuracy: {scores.mean():.2f}")

# Print computational time and accuracy
elapsed = timeit.default_timer() - start_time
print("--- Run time is %s seconds ---" % elapsed)
print(f"Accuracy on the test set: {accuracy:.2f}")

print(classification_report(y_test, y_pred, digits=3))


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000281 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4203
[LightGBM] [Info] Number of data points in the train set: 331, number of used features: 53
[LightGBM] [Info] Start training from score -1.092588
[LightGBM] [Info] Start training from score -1.101638
[LightGBM] [Info] Start training from score -1.101638
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No fu

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000305 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3468
[LightGBM] [Info] Number of data points in the train set: 265, number of used features: 53
[LightGBM] [Info] Start training from score -1.091093
[LightGBM] [Info] Start training from score -1.102393
[LightGBM] [Info] Start training from score -1.102393
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No fu

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [9]:
#Feature importance map for lightgbm model
feat_imp = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False).reset_index(drop=True)

print(feat_imp)

         feature  importance
0      Max_Tonic         492
1   kurtosis_raw         418
2       SD_Tonic         291
3      Min_Tonic         285
4     Max_Phasic         260
5        Max_raw         214
6     Mean_Tonic         189
7   skewness_raw         107
8            V40          75
9        std_raw          55
10      mean_raw          51
11       Min_raw          48
12           V29          32
13           V26          31
14            V2          22
15            V1          21
16            V7          19
17           V31          18
18           V32          18
19           V33          17
20           V20          17
21           V39          15
22           V12          13
23           V35          13
24           V10          11
25           V13           9
26            V5           8
27            V6           7
28           V28           7
29            V3           7
30           V14           7
31           V16           7
32           V34           7
33           V

In [ ]:
# If source_rows = participant ID, test on a held-out participant entirely
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, Y, groups=df['source_rows']))

X_train_g, X_test_g = X.iloc[train_idx], X.iloc[test_idx]
y_train_g, y_test_g = Y.iloc[train_idx], Y.iloc[test_idx]